In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# NOTEBOOK: PDM Vaughan Cannibalization Analysis
# CELL 1 — Source DataFrame: Playdium venues, customer grain, Food/Games labeled
# Visit = distinct (CDE_ID, VisitDateId, LocationId) — identified customers only.
# ═════════════════════════════════════════════════════════════════════════════
import pyspark.sql.functions as F
import pandas as pd
from datetime import date, timedelta

yesterday = (date.today() - timedelta(days=1)).strftime("%Y%m%d")

T_trans    = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_item     = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")

# Category sets — identical to the FIFA notebook (PDM gaming lands under TRR - AMUSEMENT codes)
games_categories = {
    "TRR - AMUSEMENT - G&E",
    "TRR - AMUSEMENT - GAMES",
    "TRR - AMUSEMENT - PHYSICAL GAMES",
    "TRR - PROMO - AMUSEMENT",
}
food_categories = {
    "MISC", "PDM - MENU FOOD - CANDY", "PDM - MENU FOOD - G&E",
    "TRR - INV ALC BEV - BOTTLE", "TRR - INV ALC BEV - CAN", "TRR - INV ALC BEV - WINE",
    "TRR - MENU ALC BEV - COCKTAIL", "TRR - MENU ALC BEV - DRAUGHT BEER",
    "TRR - MENU BEV - NA COLD BEV", "TRR - MENU BEV - NA HOT BEV",
    "TRR - MENU FOOD - APP", "TRR - MENU FOOD - DESSERT", "TRR - MENU FOOD - G&E",
    "TRR - MENU FOOD - KIDS", "TRR - MENU FOOD - PIZZA", "TRR - MENU FOOD - POUTINE",
    "TRR - MENU FOOD - SANDWICHES", "TRR - MOD MENU - ADD-ON", "TRR - MOD MENU - NO VALUE",
    "CHARITY", "COUPONS", "TRR - INV BEV - NA BEV", "TRR - MENU ALC BEV - LIQUOR",
    "TRR - MENU ALC BEV - WINE", "TRR - MENU FOOD - ENTRÉE", "TRR - MENU FOOD - SALAD",
    "VHO-LICENSED FOOD", "Box Office Surcharges and Fees", "TRR - INV ALC BEV - LIQUOR",
    "VHO-CANDY", "VIP Food - Addons", "VHO-LICENSED DRINKS", "TRR - MENU FOOD - BRUNCH",
    "TRR - MENU FOOD - G&E OLD", "VHO-BIRTHDAY", "VHO-DRINK",
}

# The four Playdium venues in scope (Playdium-only — NOT the CPX theatre rows)
PDM_VENUES = ["Playdium Vaughan", "Playdium Orion Gate", "Playdium Whitby", "Playdium Fairview"]

# ── Source DataFrame: item-line grain, carries CDE_ID + LocationId for visit tuple ──
pdm_tx = (
    T_trans
    .join(T_location, T_trans.LocationId == T_location.LocationID, "inner")
    .join(T_item, T_trans.ItemSkey == T_item.itemskey, "inner")
    .filter(F.col("Location_Name").isin(PDM_VENUES))
    .filter((F.col("VisitDateId") >= 20250101) & (F.col("VisitDateId") <= int(yesterday)))
    .filter(F.col("Item_Class_Description1").isin(list(games_categories) + list(food_categories)))
    .withColumn(
        "category_label",
        F.when(F.col("Item_Class_Description1").isin(list(games_categories)), F.lit("Games"))
         .when(F.col("Item_Class_Description1").isin(list(food_categories)),  F.lit("Food"))
    )
    .filter(F.col("category_label").isNotNull())
    .select(
        "Location_Name",
        T_trans.LocationId.alias("LocationId"),   # qualified: both tables have LocationId
        "CDE_ID", "VisitDateId",
        "category_label", "Net_Revenue","Transaction_Number"
    )
)

# ── Vaughan opening date from the location dim (don't hardcode) ──
vaughan_open = (
    T_location.filter(F.col("Location_Name") == "Playdium Vaughan")
              .select("Opening_Date").distinct().collect()
)
print("=== Playdium Vaughan Opening_Date (from cpx_d_location) ===")
print(vaughan_open, "\n")

# ── VERIFY: per venue — item rows, identified visits, customers, date span ──
print("=== Per-venue counts + date span (identified customers only for visits) ===")
(pdm_tx
   .filter(F.col("CDE_ID").isNotNull())
   .groupBy("Location_Name")
   .agg(F.count("*").alias("item_rows"),
        F.countDistinct(F.concat_ws("|", F.col("CDE_ID").cast("string"), F.col("VisitDateId").cast("string"))).alias("identified_visits"),
        F.countDistinct("CDE_ID").alias("customers"),
        F.min("VisitDateId").alias("first_date"),
        F.max("VisitDateId").alias("last_date"))
   .orderBy("Location_Name")
   .show(truncate=False))

# ── CDE_ID coverage — visits only count identified customers; this is the ceiling ──
total   = pdm_tx.count()
nonnull = pdm_tx.filter(F.col("CDE_ID").isNotNull()).count()
print(f"=== CDE_ID coverage: {nonnull:,}/{total:,} item rows = {100*nonnull/total:.1f}% identified "
      f"({100*(total-nonnull)/total:.1f}% guest/anonymous) ===")
print("Note: 'identified_visits' excludes guests → treat as a LOWER BOUND on true footfall.")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 2 — Transference (3 tiers) + Visit shift
# Boundary: Vaughan opens 2026-06-17. Post = Jun17–Jul26 2026.
# Transference lookback: incumbent customers active Jan1–Jun16 2026.
# Visit = distinct (CDE_ID, VisitDateId, LocationId), identified customers only.
# ═════════════════════════════════════════════════════════════════════════════
import pyspark.sql.functions as F

OPEN      = 20260617
POST_END  = 20260726                      # yesterday / last data date
POST      = (OPEN, POST_END)
POST_25   = (20250617, 20250726)          # same dates prior year
PRE_2026  = (20260508, 20260616)          # 40 days immediately pre-opening (matched length)
LOOKBACK  = (20260101, 20260616)          # "existing customer" definition

INCUMBENTS = ["Playdium Orion Gate", "Playdium Whitby", "Playdium Fairview"]
VAUGHAN    = "Playdium Vaughan"

ident = pdm_tx.filter(F.col("CDE_ID").isNotNull())

def _visits(df, loc, lo, hi):
    """Distinct (CDE_ID, VisitDateId) at one venue in a date range → visit-grain rows."""
    return (df.filter((F.col("Location_Name")==loc) &
                      (F.col("VisitDateId")>=lo) & (F.col("VisitDateId")<=hi))
              .select("CDE_ID", "VisitDateId").distinct())

# ── Vaughan's post-opening customers (who showed up at the new venue) ──
vaughan_cust = (ident.filter((F.col("Location_Name")==VAUGHAN) &
                             (F.col("VisitDateId")>=OPEN) & (F.col("VisitDateId")<=POST_END))
                     .select("CDE_ID").distinct())

print("="*70)
print("TRANSFERENCE — identified customers only (22.5% coverage → LOWER BOUND)")
print("="*70)

trans_rows = []
for loc in INCUMBENTS:
    # existing customers of this incumbent (active Jan1–Jun16 2026)
    existing = (ident.filter((F.col("Location_Name")==loc) &
                             (F.col("VisitDateId")>=LOOKBACK[0]) & (F.col("VisitDateId")<=LOOKBACK[1]))
                     .select("CDE_ID").distinct())
    n_existing = existing.count()

    # T1: existing incumbent customer who appears at Vaughan post-opening
    t1 = existing.join(vaughan_cust, "CDE_ID", "inner")
    n_t1 = t1.count()

    # visit counts at the incumbent, pre vs post (matched 40-day windows)
    pre_v  = (_visits(ident, loc, *PRE_2026).groupBy("CDE_ID").count()
              .withColumnRenamed("count", "pre_visits"))
    post_v = (_visits(ident, loc, *POST).groupBy("CDE_ID").count()
              .withColumnRenamed("count", "post_visits"))

    # among T1 customers: did they reduce visits to their home incumbent?
    t1_behav = (t1.join(pre_v, "CDE_ID", "left").join(post_v, "CDE_ID", "left")
                  .fillna(0, subset=["pre_visits", "post_visits"]))
    # T2: T1 who reduced OR stopped incumbent visits post-opening
    n_t2 = t1_behav.filter(F.col("post_visits") < F.col("pre_visits")).count()
    # T3: net visit-count change per shared (T1) customer
    t3 = t1_behav.agg(F.sum("post_visits").alias("post"),
                      F.sum("pre_visits").alias("pre")).collect()[0]
    net_visit_change = (t3["post"] or 0) - (t3["pre"] or 0)

    trans_rows.append({
        "Incumbent": loc.replace("Playdium ",""),
        "Existing custs (Jan–Jun16)": n_existing,
        "T1: also visited Vaughan": n_t1,
        "T1 as % of existing": round(100*n_t1/n_existing,1) if n_existing else None,
        "T2: reduced home visits": n_t2,
        "T3: net visit Δ (shared)": int(net_visit_change),
    })

import pandas as pd
tdf = pd.DataFrame(trans_rows)
print(tdf.to_string(index=False))
print("\nT1 = existing incumbent customer who also appeared at Vaughan post-opening")
print("T2 = subset of T1 who made FEWER identified visits to their home venue after opening")
print("T3 = net change in total identified visits by shared customers (neg = fewer trips to incumbent)")

# ── VISIT SHIFT — incumbents, post vs dual baseline ──
print("\n" + "="*70)
print("VISIT SHIFT — identified visits, post-opening vs baselines")
print("="*70)

vs_rows = []
for loc in INCUMBENTS + [VAUGHAN]:
    post   = _visits(ident, loc, *POST).count()
    base25 = _visits(ident, loc, *POST_25).count()      # same dates 2025
    pre26  = _visits(ident, loc, *PRE_2026).count()      # 40 days pre-opening 2026
    vs_rows.append({
        "Venue": loc.replace("Playdium ",""),
        "Post (Jun17–Jul26'26)": post,
        "Same dates '25": base25,
        "YoY Δ%": round(100*(post/base25-1),1) if base25 else None,
        "Pre-open '26 (40d)": pre26,
        "vs Pre Δ%": round(100*(post/pre26-1),1) if pre26 else None,
    })
vdf = pd.DataFrame(vs_rows)
print(vdf.to_string(index=False))
print("\nIncumbent YoY Δ% negative = fewer identified visits than last year (possible cannibalization,")
print("but NOT yet detrended — Cell 3 nets out each venue's own YoY trend before attributing to Vaughan).")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 3 — Revenue shift, detrended. Food & Games separately, per incumbent.
# ALL transactions (guests included) — not limited by CDE_ID coverage.
# Dual baseline: (a) same dates 2025 YoY, (b) detrended via each venue's own
# Jan–May 2026 YoY trend (the FIFA counterfactual-gap method).
# Post = Jun17–Jul26 2026. Vaughan excluded (no pre-history to detrend).
# ═════════════════════════════════════════════════════════════════════════════
import pyspark.sql.functions as F
import pandas as pd
import numpy as np

INCUMBENTS = ["Playdium Orion Gate", "Playdium Whitby", "Playdium Fairview"]
POST      = (20260617, 20260726)
POST_25   = (20250617, 20250726)

def _rev(loc, cat, lo, hi):
    r = (pdm_tx.filter((F.col("Location_Name")==loc) & (F.col("category_label")==cat) &
                       (F.col("VisitDateId")>=lo) & (F.col("VisitDateId")<=hi))
               .agg(F.sum(F.col("Net_Revenue").cast("double")).alias("r")).collect()[0]["r"])
    return float(r) if r is not None else 0.0

# monthly Food/Games revenue by venue+year+month for the Jan–May trend
monthly = (pdm_tx
    .withColumn("yr", (F.col("VisitDateId")/10000).cast("int"))
    .withColumn("mo", ((F.col("VisitDateId")/100)%100).cast("int"))
    .groupBy("Location_Name","yr","mo","category_label")
    .agg(F.sum(F.col("Net_Revenue").cast("double")).alias("rev"))
    .toPandas())

def _mrev(loc, cat, yr, mo):
    v = monthly[(monthly.Location_Name==loc)&(monthly.category_label==cat)&
                (monthly.yr==yr)&(monthly.mo==mo)]["rev"]
    return float(v.iloc[0]) if len(v) else np.nan

rows = []
for loc in INCUMBENTS:
    for cat in ["Food","Games"]:
        # own Jan–May 2026 YoY trend → counterfactual scaler
        pre = [(_mrev(loc,cat,2026,mo)/_mrev(loc,cat,2025,mo)-1)
               for mo in range(1,6)
               if _mrev(loc,cat,2025,mo) and not np.isnan(_mrev(loc,cat,2026,mo))]
        trend = np.mean(pre) if pre else np.nan

        post_26 = _rev(loc,cat,*POST)
        base_25 = _rev(loc,cat,*POST_25)
        raw_yoy = (post_26/base_25-1) if base_25 else np.nan

        # detrended counterfactual: what post-window SHOULD have been given own trend
        cf   = base_25*(1+trend) if not np.isnan(trend) else np.nan
        incr = post_26 - cf if not np.isnan(cf) else np.nan   # neg = below trend = possible cannibalization

        rows.append({
            "Venue": loc.replace("Playdium ",""), "Cat": cat,
            "Own trend (Jan–May)%": round(trend*100,0) if not np.isnan(trend) else None,
            "Post '26 $": round(post_26,0),
            "Same-dates '25 $": round(base_25,0),
            "Raw YoY %": round(raw_yoy*100,0) if not np.isnan(raw_yoy) else None,
            "Counterfactual $": round(cf,0) if not np.isnan(cf) else None,
            "Detrended Δ $": round(incr,0) if not np.isnan(incr) else None,
        })

rdf = pd.DataFrame(rows)
def _m(v): return "—" if pd.isna(v) else f"${v:,.0f}"
for c in ["Post '26 $","Same-dates '25 $","Counterfactual $","Detrended Δ $"]:
    rdf[c] = rdf[c].map(_m)
print("="*90)
print("REVENUE SHIFT, DETRENDED — Post-Vaughan window vs own-trend counterfactual")
print("Detrended Δ negative = venue ran BELOW its own trend after Vaughan opened (possible cannibalization)")
print("="*90)
print(rdf.to_string(index=False))
print("\nUses ALL transactions incl. guests → not limited by the 22.5% CDE_ID coverage.")
print("~6-week post window — directional, not definitive. Re-run at 90 days for a firmer read.")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 5 — Detrended VISITATION (transaction-count proxy, ALL customers incl. guests)
# Visit proxy = distinct (Transaction_Number, VisitDateId, LocationId).
# Same detrending as Cell 3: own Jan–May 2026 YoY trend → counterfactual → delta.
# NOTE: pdm_tx must carry Transaction_Number — see one-line add to Cell 1 below.
# ═════════════════════════════════════════════════════════════════════════════
import pyspark.sql.functions as F
import pandas as pd
import numpy as np

INCUMBENTS = ["Playdium Orion Gate", "Playdium Whitby", "Playdium Fairview"]
POST    = (20260617, 20260726)
POST_25 = (20250617, 20250726)

# distinct-transaction key (guest-inclusive)
TXKEY = F.concat_ws("|", F.col("Transaction_Number").cast("string"),
                         F.col("VisitDateId").cast("string"),
                         F.col("LocationId").cast("string"))

def _txn(loc, lo, hi):
    return (pdm_tx.filter((F.col("Location_Name")==loc) &
                          (F.col("VisitDateId")>=lo) & (F.col("VisitDateId")<=hi))
                  .select(TXKEY.alias("k")).distinct().count())

# monthly distinct-transaction counts for the Jan–May trend
monthly_txn = (pdm_tx
    .withColumn("yr",(F.col("VisitDateId")/10000).cast("int"))
    .withColumn("mo",((F.col("VisitDateId")/100)%100).cast("int"))
    .withColumn("k", TXKEY)
    .groupBy("Location_Name","yr","mo")
    .agg(F.countDistinct("k").alias("txns"))
    .toPandas())

def _m(loc,yr,mo):
    v = monthly_txn[(monthly_txn.Location_Name==loc)&(monthly_txn.yr==yr)&(monthly_txn.mo==mo)]["txns"]
    return float(v.iloc[0]) if len(v) else np.nan

rows=[]
for loc in INCUMBENTS:
    pre=[(_m(loc,2026,mo)/_m(loc,2025,mo)-1) for mo in range(1,6)
         if _m(loc,2025,mo) and not np.isnan(_m(loc,2026,mo))]
    trend=np.mean(pre) if pre else np.nan
    post=_txn(loc,*POST); base=_txn(loc,*POST_25)
    raw=(post/base-1) if base else np.nan
    cf=base*(1+trend) if not np.isnan(trend) else np.nan
    delta=post-cf if not np.isnan(cf) else np.nan
    rows.append({
        "Venue":loc.replace("Playdium ",""),
        "Own trend %":round(trend*100,0) if not np.isnan(trend) else None,
        "Post txns":int(post),
        "Same-dates '25":int(base),
        "Raw YoY %":round(raw*100,0) if not np.isnan(raw) else None,
        "Counterfactual":round(cf,0) if not np.isnan(cf) else None,
        "Detrended Δ txns":round(delta,0) if not np.isnan(delta) else None,
    })
df=pd.DataFrame(rows)
print("="*80)
print("DETRENDED VISITATION — distinct-transaction proxy, ALL customers (guests incl.)")
print("Detrended Δ negative = fewer transactions than own trend predicts → possible cannibalization")
print("="*80)
print(df.to_string(index=False))
print("\nProxy = distinct transactions, not exact visits (one visit may split into 2 checks),")
print("but bias is consistent over time so the detrended comparison holds. ~6-week window.")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# FOUR CHARTS — one per analytical lens. Run as one cell; produces 4 figures.
# ═════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np

venues = ["Orion Gate", "Whitby", "Fairview"]
GREEN, RED, ORANGE, BLUE, PURPLE = "#2E7D32", "#C62828", "#FF9800", "#2196F3", "#9C27B0"

def _label_bars(ax, bars, money=True, unit=""):
    for bar in bars:
        h = bar.get_height()
        txt = f"${h/1000:+.0f}K" if money else f"{h:+.0f}{unit}"
        ax.annotate(txt, xy=(bar.get_x()+bar.get_width()/2, h),
                    xytext=(0, 4 if h>=0 else -14), textcoords="offset points",
                    ha="center", fontsize=9, fontweight="bold",
                    color=GREEN if h>=0 else RED)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 1 — TRANSFERENCE: how many existing customers switched
# ─────────────────────────────────────────────────────────────────────────────
t1 = [67, 52, 141]      # also visited Vaughan
t2 = [16, 9, 22]        # reduced home visits (true switch)
bases = [6334, 7290, 6238]

x = np.arange(len(venues)); w = 0.38
fig1, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w/2, t1, w, label="Visited Vaughan (T1)", color=BLUE, edgecolor="white")
b2 = ax.bar(x + w/2, t2, w, label="Reduced home visits (T2 = true switch)", color=RED, edgecolor="white")
_label_bars(ax, b1, money=False); _label_bars(ax, b2, money=False)
ax.set_title("1 · Customer Transference to Vaughan\n"
             f"Out of {bases[0]:,}–{bases[1]:,} existing customers per venue — switchers are a rounding error",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Identified customers")
ax.set_xticks(x); ax.set_xticklabels(venues, fontsize=11)
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True); ax.legend(fontsize=9, loc="upper left")
ax.text(0.5, -0.16, "Identified customers only (22.5% coverage) → lower bound. Even ×4 for guests, immaterial vs bases.",
        transform=ax.transAxes, ha="center", fontsize=8, style="italic", color="#555")
plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# CHART 2 — VISITATION (footfall): detrended transaction count vs own trend
# ─────────────────────────────────────────────────────────────────────────────
txn_delta = [-710, -361, 46]   # detrended Δ transactions (Cell 5, all customers incl. guests)
fig2, ax = plt.subplots(figsize=(11, 6))
colors = [RED if v < 0 else GREEN for v in txn_delta]
bars = ax.bar(venues, txn_delta, 0.55, color=colors, edgecolor="white")
_label_bars(ax, bars, money=False)
ax.axhline(0, color="#333", lw=1.5)
ax.axhspan(ax.get_ylim()[0], 0, alpha=0.04, color="red", zorder=0)
ax.set_title("2 · Footfall vs Own Trend (all customers incl. guests)\n"
             "Detrended transaction count · small dips at Orion/Whitby, within 6-week noise",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Transactions above / below own trend")
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
ax.text(0.5, -0.14, "Below zero = fewer transactions than the venue's own Jan–May trend predicts. Not limited by CDE_ID coverage.",
        transform=ax.transAxes, ha="center", fontsize=8, style="italic", color="#555")
plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# CHART 3 — REVENUE SHIFT: detrended Food vs Games (the headline)
# ─────────────────────────────────────────────────────────────────────────────
food_delta  = [-7088, 10867, -11233]
games_delta = [-5802, 44087,  41471]
fig3, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w/2, food_delta,  w, label="Food",  color=ORANGE, edgecolor="white")
b2 = ax.bar(x + w/2, games_delta, w, label="Games (Vaughan's direct category)", color=BLUE, edgecolor="white")
_label_bars(ax, b1); _label_bars(ax, b2)
ax.axhline(0, color="#333", lw=1.5)
ax.axhspan(ax.get_ylim()[0], 0, alpha=0.04, color="red", zorder=0)
ax.set_title("3 · Detrended Revenue vs Own Trend  (THE HEADLINE)\n"
             "Games — what Vaughan competes for — runs ABOVE trend at Whitby & Fairview",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Revenue above / below own trend ($)")
ax.set_xticks(x); ax.set_xticklabels(venues, fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"${v/1000:+.0f}K"))
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True); ax.legend(fontsize=10, loc="upper left")
plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# CHART 4 — THE SIGNATURE TEST: do visits AND revenue fall together?
# Cannibalization requires BOTH negative. Plot each venue as a point.
# ─────────────────────────────────────────────────────────────────────────────
total_rev_delta = [food_delta[i] + games_delta[i] for i in range(3)]  # combined detrended $
fig4, ax = plt.subplots(figsize=(9, 8))
ax.axhline(0, color="#333", lw=1.2); ax.axvline(0, color="#333", lw=1.2)
# shade the danger quadrant (both negative) = cannibalization
ax.axhspan(ax.get_ylim()[0] if False else -60000, 0, xmin=0, xmax=0.5, alpha=0.06, color="red")
ax.fill_between([-800, 0], -60000, 0, alpha=0.07, color="red", zorder=0)
for i, v in enumerate(venues):
    ax.scatter(txn_delta[i], total_rev_delta[i], s=260, zorder=5,
               color=[RED, GREEN, GREEN][i], edgecolor="black", lw=1.5)
    ax.annotate(v, (txn_delta[i], total_rev_delta[i]), xytext=(10, 8),
                textcoords="offset points", fontsize=11, fontweight="bold")
ax.set_xlim(-800, 200); ax.set_ylim(-40000, 90000)
ax.set_xlabel("Footfall vs trend (transactions)  →  right = more visits")
ax.set_ylabel("Revenue vs trend ($)  →  up = more money")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"${v/1000:+.0f}K"))
ax.set_title("4 · The Cannibalization Signature Test\n"
             "Cannibalization = BOTH fall (red quadrant, bottom-left). No venue is there.",
             fontsize=12, fontweight="bold")
ax.text(-400, -30000, "CANNIBALIZATION\nZONE\n(empty)", ha="center", fontsize=11,
        fontweight="bold", color=RED, alpha=0.6)
ax.grid(alpha=0.25); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# FAIRVIEW CANNIBALIZATION — same method as Vaughan, retargeted.
# Fairview = new opening (Nov 2024). Incumbents = Orion Gate, Whitby (GTA).
# Self-checks for 2024 data; if absent, the YoY baseline can't be built and it says so.
# ═════════════════════════════════════════════════════════════════════════════
import pyspark.sql.functions as F
import pandas as pd, numpy as np

# --- opening date from the dim ---
fv_open = (T_location.filter(F.col("Location_Name")=="Playdium Fairview")
                     .select("Opening_Date").distinct().collect())
print("Fairview Opening_Date:", fv_open)
OPEN = int(fv_open[0]["Opening_Date"].strftime("%Y%m%d")) if fv_open and fv_open[0]["Opening_Date"] else None

# --- does pre-opening data exist? ---
earliest = pdm_tx.agg(F.min("VisitDateId").alias("e")).collect()[0]["e"]
print(f"Earliest data: {earliest} | Fairview opened: {OPEN}")
HAS_PRE = OPEN is not None and earliest is not None and earliest < OPEN
print("Pre-opening data available:" , HAS_PRE,
      "" if HAS_PRE else "  → YoY/pre-post baseline CANNOT be built; transference + geo-control only.\n")

INCUMBENTS = ["Playdium Orion Gate", "Playdium Whitby"]
FAIRVIEW   = "Playdium Fairview"

# 40-day windows around opening (if pre exists)
def _yyyymmdd(base, delta_days):
    from datetime import datetime, timedelta
    d = datetime.strptime(str(base), "%Y%m%d") + timedelta(days=delta_days)
    return int(d.strftime("%Y%m%d"))

# ── TRANSFERENCE (works regardless of 2024 data — customer-overlap question) ──
ident = pdm_tx.filter(F.col("CDE_ID").isNotNull())
LOOKBACK = ( _yyyymmdd(OPEN,-180), _yyyymmdd(OPEN,-1) ) if OPEN else None
POST     = ( OPEN, _yyyymmdd(OPEN,40) ) if OPEN else None

if OPEN:
    fv_cust = (ident.filter((F.col("Location_Name")==FAIRVIEW)&
                            (F.col("VisitDateId")>=POST[0])&(F.col("VisitDateId")<=POST[1]))
                    .select("CDE_ID").distinct())
    rows=[]
    for loc in INCUMBENTS:
        existing=(ident.filter((F.col("Location_Name")==loc)&
                               (F.col("VisitDateId")>=LOOKBACK[0])&(F.col("VisitDateId")<=LOOKBACK[1]))
                       .select("CDE_ID").distinct())
        ne=existing.count(); t1=existing.join(fv_cust,"CDE_ID","inner").count()
        rows.append({"Incumbent":loc.replace("Playdium ",""),
                     "Existing custs (180d pre)":ne,
                     "T1: also visited Fairview":t1,
                     "T1 %":round(100*t1/ne,1) if ne else None})
    print("=== TRANSFERENCE (identified only, 22.5% coverage → lower bound) ===")
    print(pd.DataFrame(rows).to_string(index=False), "\n")

# ── DETRENDED REVENUE SHIFT (only if pre-opening + prior-year data exist) ──
if HAS_PRE:
    def _rev(loc,cat,lo,hi):
        r=(pdm_tx.filter((F.col("Location_Name")==loc)&(F.col("category_label")==cat)&
                         (F.col("VisitDateId")>=lo)&(F.col("VisitDateId")<=hi))
                 .agg(F.sum(F.col("Net_Revenue").cast("double")).alias("r")).collect()[0]["r"])
        return float(r) if r else 0.0
    monthly=(pdm_tx.withColumn("yr",(F.col("VisitDateId")/10000).cast("int"))
                   .withColumn("mo",((F.col("VisitDateId")/100)%100).cast("int"))
                   .groupBy("Location_Name","yr","mo","category_label")
                   .agg(F.sum(F.col("Net_Revenue").cast("double")).alias("rev")).toPandas())
    def _mrev(loc,cat,yr,mo):
        v=monthly[(monthly.Location_Name==loc)&(monthly.category_label==cat)&
                  (monthly.yr==yr)&(monthly.mo==mo)]["rev"]
        return float(v.iloc[0]) if len(v) else np.nan

    open_yr, open_mo = OPEN//10000, (OPEN//100)%100
    POST_CUR = (OPEN, _yyyymmdd(OPEN,40))
    POST_PRI = (_yyyymmdd(OPEN,-365), _yyyymmdd(OPEN,-325))  # same 40d, prior year
    rows=[]
    for loc in INCUMBENTS:
        for cat in ["Food","Games"]:
            pre=[(_mrev(loc,cat,open_yr, mo)/_mrev(loc,cat,open_yr-1,mo)-1)
                 for mo in range(1,open_mo)
                 if _mrev(loc,cat,open_yr-1,mo) and not np.isnan(_mrev(loc,cat,open_yr,mo))]
            trend=np.mean(pre) if pre else np.nan
            cur=_rev(loc,cat,*POST_CUR); pri=_rev(loc,cat,*POST_PRI)
            cf=pri*(1+trend) if not np.isnan(trend) else np.nan
            delta=cur-cf if not np.isnan(cf) else np.nan
            rows.append({"Venue":loc.replace("Playdium ",""),"Cat":cat,
                         "Trend %":round(trend*100,0) if not np.isnan(trend) else None,
                         "Post $":round(cur,0),"Prior-yr $":round(pri,0),
                         "Counterfactual $":round(cf,0) if not np.isnan(cf) else None,
                         "Detrended Δ $":round(delta,0) if not np.isnan(delta) else None})
    print("=== DETRENDED REVENUE SHIFT (Δ<0 = below trend = possible cannibalization) ===")
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("=== DETRENDED REVENUE SHIFT: SKIPPED — no pre-opening/prior-year data for Fairview ===")
    print("Fairview opened before the data window starts. Transference above is the only")
    print("available read; for a revenue shift you'd need 2024 data loaded.")

In [0]:
# ═════════════════════════════════════════════════════════════════════════════
# PDM VAUGHAN CANNIBALIZATION — single 2×2 exhibit, Cineplex palette
# ═════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter

# ── Cineplex palette ──
NAVY, NAVY_60, GOLD = "#0A2240", "#5A6E88", "#C8A24B"
SLATE, SLATE_L, SOFTRED = "#8895A7", "#DCE1E8", "#B15A54"
INK, GRIDCOL = "#1C2A3A", "#E8ECF1"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11, "text.color": INK,
    "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.edgecolor": "#C9D2DC", "figure.facecolor": "white", "axes.facecolor": "white",
})
venues = ["Orion Gate", "Whitby", "Fairview"]
K = FuncFormatter(lambda v,_: f"${v/1000:,.0f}K")
x = np.arange(3); w = 0.36

def _title(ax, kicker, headline):
    ax.text(0, 1.20, kicker, transform=ax.transAxes, fontsize=9.5, color=GOLD, fontweight="bold")
    ax.text(0, 1.10, headline, transform=ax.transAxes, fontsize=13, color=NAVY, fontweight="bold")

def _clean(ax):
    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_color("#C9D2DC")
    ax.grid(axis="y", color=GRIDCOL, lw=1); ax.set_axisbelow(True); ax.tick_params(length=0)

fig, axes = plt.subplots(2, 2, figsize=(17, 12))
fig.subplots_adjust(hspace=0.50, wspace=0.20, top=0.86, bottom=0.09, left=0.07, right=0.96)
fig.suptitle("Did Playdium Vaughan cannibalize nearby venues?  No — the incumbents held or grew.",
             fontsize=17, fontweight="bold", color=NAVY, x=0.07, ha="left", y=0.985)

# ── (0,0) Transference — no legend, direct labels ──
ax = axes[0,0]
base=[6334,7290,6238]; t1=[67,52,141]; t2=[16,9,22]
y=np.arange(3)[::-1]
ax.barh(y, base, 0.62, color=SLATE_L, zorder=2)
ax.barh(y, t1,  0.62, color=NAVY_60, zorder=3)
ax.barh(y, t2,  0.62, color=NAVY,    zorder=4)
for i in range(3):
    ax.text(base[i]+110, y[i], f"{base[i]:,} customers", va="center",
            fontsize=10.5, color=NAVY_60, fontweight="bold")
    ax.text(t2[i]+110, y[i]-0.30, f"{t2[i]} switched", va="center",
            fontsize=10, color=NAVY, fontweight="bold")
ax.set_yticks(y); ax.set_yticklabels(venues, fontsize=12, fontweight="bold")
ax.set_xlim(0, 8600); ax.set_xlabel("Customers", fontsize=10)
ax.spines[["top","right"]].set_visible(False); ax.tick_params(length=0)
# one-time inline key, top-left, out of the way
ax.text(0.0, -0.24,
        "Light = existing · Mid = tried Vaughan · Dark = shifted visits    |    "
        "Identified customers only (22.5% coverage) → lower bound",
        transform=ax.transAxes, fontsize=8.5, color=NAVY_60, style="italic")
_title(ax, "CUSTOMER TRANSFERENCE", "Almost no one switched to Vaughan")


# ── (0,1) Footfall ──
ax = axes[0,1]
exp=[8174,9384,7316]; act=[7464,9023,7362]
ax.bar(x-w/2, exp, w, color=SLATE, label="Expected", zorder=2)
ax.bar(x+w/2, act, w, color=[SOFTRED if act[i]<exp[i] else GOLD for i in range(3)], label="Actual", zorder=3)
for i in range(3):
    d=act[i]-exp[i]
    ax.text(x[i]+w/2, act[i]+160, f"{d:+,}", ha="center", fontsize=10.5, fontweight="bold",
            color=SOFTRED if d<0 else NAVY)
ax.set_xticks(x); ax.set_xticklabels(venues, fontsize=12, fontweight="bold")
ax.set_ylabel("Transactions · 6 wks", fontsize=10); ax.set_ylim(0, 10800)
_clean(ax); ax.legend(loc="upper right", fontsize=8.5, frameon=False)
_title(ax, "FOOTFALL", "Visits stayed close to expected")

# ── (1,0) Revenue — Food & Games. Fill = category (ALWAYS). Ghost = expected. ──
ax = axes[1,0]
FOOD, GAMES = GOLD, NAVY
exp_rev = {"Food":[76223,84109,53603], "Games":[417466,444282,261240]}
act_rev = {"Food":[69135,94976,42371], "Games":[411664,488369,302711]}
gw = 0.30
pos = {"Food": x-gw/2-0.02, "Games": x+gw/2+0.02}
catcol = {"Food":FOOD, "Games":GAMES}
for cat in ["Food","Games"]:
    p, c = pos[cat], catcol[cat]
    # expected = hatched ghost outline in the category color
    ax.bar(p, exp_rev[cat], gw, facecolor="white", edgecolor=c, lw=1.4, hatch="////", zorder=2)
    # actual = SOLID category color, ALWAYS (never recolored)
    ax.bar(p, act_rev[cat], gw, color=c, zorder=3)
    for i in range(3):
        d = act_rev[cat][i]-exp_rev[cat][i]
        ax.text(p[i], max(act_rev[cat][i],exp_rev[cat][i])+9000, f"{d/1000:+.0f}K",
                ha="center", fontsize=9.5, fontweight="bold", color=SOFTRED if d<0 else "#2E7D32")
ax.set_xticks(x); ax.set_xticklabels(venues, fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(K); ax.set_ylabel("Revenue · 6 wks", fontsize=10); ax.set_ylim(0,575000)
_clean(ax)
from matplotlib.patches import Patch
leg=[Patch(facecolor=FOOD,label="Food"), Patch(facecolor=GAMES,label="Games"),
     Patch(facecolor="white",edgecolor="#555",hatch="////",label="Expected")]
ax.legend(handles=leg, loc="upper center", fontsize=8, frameon=False, ncol=3, bbox_to_anchor=(0.5,1.02))
_title(ax, "REVENUE — FOOD & GAMES", "Gaming grew; only Fairview food dipped")

# ── (1,1) Verdict quadrant ──
ax = axes[1,1]
txn_d=[-710,-361,46]; rev_d=[-12890,54954,30238]
ax.fill_between([-900,0], -30000, 0, color=SOFTRED, alpha=0.10, zorder=0)
ax.axhline(0, color=NAVY, lw=1.1); ax.axvline(0, color=NAVY, lw=1.1)
for i,v in enumerate(venues):
    inzone = txn_d[i]<0 and rev_d[i]<0
    ax.scatter(txn_d[i], rev_d[i], s=300, color=SOFTRED if inzone else GOLD, edgecolor=NAVY, lw=1.6, zorder=5)
    ax.annotate(v, (txn_d[i],rev_d[i]), xytext=(12,9), textcoords="offset points",
                fontsize=11.5, fontweight="bold", color=NAVY)
ax.set_xlim(-900,260); ax.set_ylim(-30000,75000); ax.yaxis.set_major_formatter(K)
ax.set_xlabel("Visits vs expected  →", fontsize=10); ax.set_ylabel("Revenue vs expected  →", fontsize=10)
ax.text(-450,-21000,"CANNIBALIZATION ZONE", ha="center", fontsize=9.5, fontweight="bold", color=SOFTRED)
_clean(ax); ax.grid(True, color=GRIDCOL, lw=1)
_title(ax, "THE VERDICT", "Both must fall — only Orion clips it")

fig.text(0.07, 0.035,
         "6-week window (Jun 17–Jul 26 2026) vs each venue's own pre-Vaughan trend. Directional; recheck at 90 days. "
         "Transference = identified customers only (22.5% coverage → lower bound).",
         ha="left", fontsize=9.5, color=NAVY_60, style="italic")

for ax in axes.flat:
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontweight("bold")
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight("bold")
plt.show()